### With Structure LLM give Response In a particular format 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")
from IPython.display import display, Markdown

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

### Pydantic
It is a data validation library built on top of standard Python type hints. In AI engineering—especially with frameworks like LangChain, LangGraph, and tool calling—BaseModel and Field are the absolute building blocks for structured communication.

### 1. What is BaseModel?
BaseModel is a class you inherit from to create a strictly typed blueprint for data. It forces incoming data to match specific types, and it automatically converts (coerces) data when it can.

### 2. What is Field?
While type hints (like int or str) tell Pydantic what type of data to expect, Field allows you to inject extra rules, limits, and descriptions to that specific attribute.

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title Of the movie")
    year:int=Field(description="Year Movie was released")
    director:str=Field(description="Director of the movie")
    rating:float=Field(description="IBM rating of the movie")

In [5]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model = "google_genai:gemini-3.1-flash-lite")
response = model.invoke("tell me about the Game of thrones")
response

AIMessage(content=[{'type': 'text', 'text': '*Game of Thrones* (GoT) is a landmark fantasy drama series that aired on HBO from 2011 to 2019. Based on George R.R. Martin’s best-selling novel series, *A Song of Ice and Fire*, the show became a global cultural phenomenon known for its massive scale, shocking plot twists, and morally ambiguous characters.\n\nHere is a breakdown of what makes the show significant:\n\n### 1. The Setting\nThe story takes place primarily on the continent of **Westeros**, a land divided into Seven Kingdoms. The political landscape is governed by noble families (Houses) who vie for control of the **Iron Throne**, the seat of power in the capital city, King’s Landing.\n\nWhile the humans fight for political control, two existential threats loom in the background:\n*   **The Winter:** A cycle where seasons last for years, and a "Long Night" is approaching.\n*   **The White Walkers:** An ancient, undead army from the frozen North that threatens to wipe out all life

In [7]:
model_with_strucutre = model.with_structured_output(Movie)
response = model_with_strucutre.invoke("tell me about the Game of thrones")
response

Movie(title='Game of Thrones', year=2011, director='David Benioff and D.B. Weiss', rating=9.2)

### Nested Structure

In [10]:
class Actor(BaseModel):
    name:str = Field("Name Of the Actor")
    character:str = Field("Character They Played")

class MovieDetails(BaseModel):
    title:str = Field("Title of the Series/Movie")
    year:int = Field("Year The movie/series released")
    genres:list[str] = Field("Genres of the Movie/Series")
    budget:float = Field("Budget of the movie/series")
    actors:list[Actor] = Field("Actors of the Movie/Series")

model_with_movie_details = model.with_structured_output(MovieDetails)
response = model_with_movie_details.invoke("tell me about the Game of thrones")
response

MovieDetails(title='Game of Thrones', year=2011, genres=['Fantasy', 'Drama', 'Action'], budget=100000000.0, actors=[Actor(name='Emilia Clarke', character='Daenerys Targaryen'), Actor(name='Kit Harington', character='Jon Snow'), Actor(name='Peter Dinklage', character='Tyrion Lannister')])

### TypeDict

<b>TypedDict</b>
As we went over, this defines a blueprint for a standard Python dictionary. It promises that any dictionary matching this schema will contain the keys title, year, director, and rating.

<b>Annotated</b>
Annotated is a built-in Python typing utility that lets you attach extra metadata to a standard type hint without breaking Python's internal rules. It follows this exact syntax layout:

... (The Ellipsis): In this specific context, the ellipsis acts as a placeholder or a marker indicating that this field is strictly required (there is no default value provided).
### Comparison: TypedDict vs. Pydantic (BaseModel)

| Feature | TypedDict (Standard Python) | Pydantic (BaseModel) |
| :--- | :--- | :--- |
| **Data Type** | Creates a standard Python `dict` | Creates a custom Python Object |
| **Validation Time** | **Static analysis only** (checked by VS Code/Cursor/IDE before running) | **Runtime** (checked actively while your app is running) |
| **Type Coercion** | No (ignores wrong types at runtime) | **Yes** (automatically converts `"123"` to `123`) |
| **Access Syntax** | Square brackets: `data["name"]` | Dot notation: `data.name` |
| **Framework Use Case** | Used extensively to define **LangGraph States** | Used to define **Tools and Structured Outputs** |

In [11]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie With Details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response


{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [14]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

### Work with Create Agent

In [17]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field

class contactInfo(BaseModel):
    """ Contact Information for a person"""
    name:str
    email:str
    phone:str

agent = create_agent(
    model = "google_genai:gemini-3.1-flash-lite",
    response_format = contactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

contactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

### Comparison: TypedDict vs. Dataclasses vs. Pydantic

| Feature | TypedDict | Dataclasses (Standard Python) | Pydantic (BaseModel) |
| :--- | :--- | :--- | :--- |
| **Data Type** | Standard Python `dict` | Custom Python Object | Custom Python Object |
| **Boilerplate Built-in** | Yes | **Yes** (Generates `__init__`, `__repr__`) | **Yes** (Full object management) |
| **Validation Time** | Static analysis only (IDE) | **Static analysis only** (IDE) | **Runtime** (Active checking) |
| **Type Coercion** | No | No | **Yes** (Converts `"123"` to `123`) |
| **Access Syntax** | `data["name"]` | Dot notation: `data.name` | Dot notation: `data.name` |
| **Performance Cost** | Extremely low | Very low | Slightly higher (due to parsing checks) |
| **AI Framework Use Case**| LangGraph State Maps | Simple data storage objects | **Tools and Structured Outputs** |

In [18]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model = "google_genai:gemini-3.1-flash-lite",
    response_format = contactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']


contactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')